# Mk33: train the first model and test PALM against AREX-2's masking (Kaggle)

The paper's main technical claim is that **Progress-Aware Loss Masking (PALM)** trains a better long-horizon agent
than AREX-2's masking or training on every turn. This notebook runs that test (ablation A1) at a scale that fits a
free Colab T4:

1. **Teacher runs.** A teacher model works on training versions of the bench tasks (data seeds 1 to 3).
2. **Select and weight.** Runs that improved are kept. Each turn is weighted three ways: PALM, AREX-2 (zero for
   redundant, empty and polling turns, full weight otherwise) and uniform (every turn).
3. **Train three students.** Qwen2.5-1.5B-Instruct with LoRA, once per arm. Same data, same settings, only the loss mask differs.
4. **Test.** Base model vs uniform vs AREX vs PALM on the held-out tasks (seed 0, never seen in training), with paired
   95% intervals.

**Time.** Roughly 15 to 20 GPU hours in total, so about two Kaggle sessions. Every step skips work that is already done.

**What a result means.** This is small: 1.5B model, two task types, a few dozen trajectories. A clear win for PALM
here is evidence worth reporting; no difference is also a result. Either way, report it with the intervals.

## How to run on Kaggle

1. Kaggle account with a **verified phone number** (Settings > Phone verification); needed for GPUs and internet.
2. In this notebook's right-hand panel: **Accelerator: GPU T4 x2** (one GPU is used) and **Internet: on**.
3. Click **Save Version**, choose **Save & Run All (Commit)**, and save. Kaggle runs the whole notebook in the background for up to 12 hours; you can close the browser.
4. When it finishes, the results are in the version's **Output** tab, and the report is printed near the end of the log.

**Continuing in a new session.** The notebook stops starting new runs about an hour before Kaggle's 12-hour limit, so every finished run is saved. To continue: **Add Input** (right panel) > **Your Work** > this notebook, pick the previous version, then Save Version again. The notebook copies the earlier results in first and skips everything already finished.

Kaggle gives about 30 free GPU hours per week; the remaining quota is shown in Settings.

## 1. GPU, results folder and install

In [ ]:
import time, os
START = time.time()
SESSION_HOURS = 11.0   # Kaggle stops commits at 12 h; leave a margin so the output is saved


def hours_left():
    return SESSION_HOURS - (time.time() - START) / 3600


def have_time(hours_needed):
    """True if there is time to start a step that may take up to `hours_needed` hours."""
    ok = hours_left() >= hours_needed
    if not ok:
        print(f"Stopping here to save results ({hours_left():.1f} h left in this session). "
              "Continue in a new version with this version's output added as input.")
    return ok

import torch
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 x2 in the right panel."
print(f"GPU: {torch.cuda.get_device_name()}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

import glob, shutil
OUT_DIR = "/kaggle/working/train_palm"
# Results from an earlier version of this notebook, attached with Add Input > Your Work.
previous = sorted(glob.glob("/kaggle/input/*/train_palm")) + sorted(glob.glob("/kaggle/input/*/*/train_palm"))
for p in previous:
    print("continuing from", p)
    shutil.copytree(p, OUT_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("results folder:", OUT_DIR, "| finished runs so far:",
      len(glob.glob(f"{OUT_DIR}/**/result.json", recursive=True)))
ROOT = OUT_DIR

In [ ]:
%cd /tmp
!test -d Mk33 || git clone -q https://github.com/dulguuntsatsralt-cell/Mk33
%cd /tmp/Mk33
!git pull -q
!pip install -q -e '.[mk33-kv,mk33-lora,dev]'
!git log --oneline -1

## 2. Settings

**Teacher.** `local` runs Qwen2.5-3B-Instruct on this GPU: free, but slow. For an API teacher, set `TEACHER` to the
model name and `TEACHER_BASE_URL` to an OpenAI-compatible endpoint, and add your key as a Kaggle secret named `MODEL_API_KEY` (**Add-ons > Secrets** in the editor menu, then turn it on for this notebook). Use an **open-weight model whose license allows training on its
outputs** (Qwen models do). Closed models such as GPT, Claude or Gemini forbid this in their terms.

In [ ]:
TEACHER = "local"            # or an API model name, e.g. a Qwen model from your provider's list
TEACHER_BASE_URL = ""        # e.g. "https://openrouter.ai/api/v1" for an API teacher
STUDENT = "Qwen/Qwen2.5-1.5B-Instruct"

TRAIN_SEEDS = [1, 2, 3]      # training data; seed 0 is held out for testing
TASKS = "knapsack,regression"
TEACHER_REPEATS = 2          # teacher runs per training environment
EVAL_REPEATS = 3
TAU = 0.5                    # keep teacher runs that reached at least this normalized score

BUDGET = "--modes rlc --window 8000 --max-rounds 8 --max-tool-calls 40 --max-minutes 25"

## 3. Environments

Generation is seeded, so every session rebuilds identical environments. Seeds 1 to 3 have different data from
seed 0, so nothing the students train on appears in the test.

In [ ]:
import json, glob
for s in TRAIN_SEEDS:
    !mk33 bench-prepare --out bench/train/seed{s} --tasks {TASKS} --seed {s} | tail -1
!mk33 bench-prepare --out bench/eval --tasks {TASKS} --seed 0 | tail -1

with open("bench/train/envs.jsonl", "w") as out:
    for s in TRAIN_SEEDS:
        out.write(open(f"bench/train/seed{s}/envs.jsonl").read())
ids = [json.loads(l)["env_id"] for l in open("bench/train/envs.jsonl")]
print("training environments:", ids)

## 4. Teacher runs

In [ ]:
TEACHER_OUT = f"{ROOT}/teacher"
for rep in range(1, TEACHER_REPEATS + 1):     # one repeat (6 runs) per call
    if not have_time(2.5):
        break
    if TEACHER == "local":
        !mk33 bench-run --backend hf --model Qwen/Qwen2.5-3B-Instruct --dtype float16 \
            --envs bench/train/envs.jsonl --out {TEACHER_OUT} --repeats {rep} {BUDGET} 2>&1 | grep -v -i warning | tail -3
    else:
        from kaggle_secrets import UserSecretsClient
        os.environ["MODEL_API_KEY"] = UserSecretsClient().get_secret("MODEL_API_KEY")
        !mk33 bench-run --model {TEACHER} --base-url {TEACHER_BASE_URL} --workers 4 \
            --envs bench/train/envs.jsonl --out {TEACHER_OUT} --repeats {rep} {BUDGET} | tail -3
!head -30 {TEACHER_OUT}/report.md

## 5. Select runs and build training windows

`mk33 select` keeps runs that reached `TAU`, lasted at least 3 rounds and improved after round 1, at most 2 per
environment. Both arms use the same selected runs; only the weights differ. If very few runs are accepted, lower
`TAU` in section 2 and run this section again.

In [ ]:
DATA = f"{ROOT}/data"
os.makedirs(DATA, exist_ok=True)
!mk33 bench-collect --out {TEACHER_OUT} --to {DATA}/trajs.jsonl

# This run's selection threshold, on top of the experiment config.
open("/tmp/palm_run.yaml", "w").write(
    f"inherit: /tmp/Mk33/configs/mk33/colab/palm.yaml\nselection: {{tau: {{ml: {TAU}, judge: 0.95}}}}\n")
!mk33 select --config /tmp/palm_run.yaml --trajs {DATA}/trajs.jsonl --out {DATA}/accepted.jsonl
n_accepted = sum(1 for l in open(f"{DATA}/accepted.jsonl") if l.strip())
print(f"accepted teacher runs: {n_accepted}")
if n_accepted < 4:
    raise RuntimeError(f"Only {n_accepted} teacher runs passed selection. Lower TAU in section 2 (for example to 0.3), "
                     "or add teacher runs (raise TEACHER_REPEATS), then run sections 2 to 5 again.")

!mk33 build-sft --config configs/mk33/colab/palm.yaml --trajs {DATA}/accepted.jsonl --out {DATA}/windows_palm.jsonl
!mk33 build-sft --config configs/mk33/colab/uniform.yaml --trajs {DATA}/accepted.jsonl --out {DATA}/windows_uniform.jsonl
!mk33 build-sft --config configs/mk33/colab/arex.yaml --trajs {DATA}/accepted.jsonl --out {DATA}/windows_arex.jsonl
!mk33 weights --config configs/mk33/colab/palm.yaml --trajs {DATA}/accepted.jsonl --out {DATA}/palm_weights.jsonl | head -30

## 6. Train the three students

LoRA (rank 16) on Qwen2.5-1.5B-Instruct in float16, 3 epochs. Each arm saves its adapter and a merged model under
`checkpoints/` on Drive. An arm that already has a merged model is skipped.

In [ ]:
CKPT = f"{ROOT}/checkpoints"
for arm in ("palm", "arex", "uniform"):
    if os.path.exists(f"{CKPT}/{arm}/merged/config.json"):
        print(f"{arm}: already trained"); continue
    if not have_time(1.5):
        break
    print(f"=== training {arm} ===")
    !mk33 train --config configs/mk33/colab/{arm}.yaml --train-files {DATA}/windows_{arm}.jsonl --output-dir {CKPT}/{arm} 2>&1 | grep -v -i warning

## 7. Test on held-out tasks

Each model runs the same held-out tasks with the same budget. Runs are ordered by repeat, so partial results stay
balanced across models.

In [ ]:
MODELS = {"base": STUDENT, "uniform": f"{CKPT}/uniform/merged", "arex": f"{CKPT}/arex/merged",
          "palm": f"{CKPT}/palm/merged"}
for rep in range(1, EVAL_REPEATS + 1):
    for name, path in MODELS.items():
        if not have_time(1.0):
            break
        if name != "base" and not os.path.exists(f"{path}/config.json"):
            print(f"{name}: not trained yet, skipping"); continue
        print(f"=== repeat {rep}: {name} ({hours_left():.1f} h left) ===")
        !mk33 bench-run --backend hf --model "{path}" --dtype float16 --envs bench/eval/envs.jsonl \
            --out {ROOT}/eval/{name} --repeats {rep} {BUDGET} 2>&1 | grep -v -i warning | tail -3

## 8. Results

Paired by task and repeat. "real difference?" is yes only when the 95% interval excludes zero. The comparison that
tests PALM against AREX-2 is **arex − palm** (negative means PALM is better); **uniform − base** shows what training
on teacher runs does at all.

In [ ]:
runs = ",".join(f"{n}={ROOT}/eval/{n}" for n in MODELS)
!mk33 bench-compare --runs {runs} --mode rlc --save {ROOT}/comparison.md

In [ ]:
import matplotlib.pyplot as plt
from mk33.bench.report import load_rows

fig, ax = plt.subplots(figsize=(6, 4))
for i, name in enumerate(MODELS):
    rows = [r for r in load_rows(f"{ROOT}/eval/{name}") if not r.get("error") and r["mode"] == "rlc"]
    xs = [r["best"] for r in rows]
    if not xs: continue
    ax.bar(i, sum(xs) / len(xs), color={"palm": "#3b5bdb", "arex": "#e8590c"}.get(name, "#999"), zorder=1)
    ax.scatter([i] * len(xs), xs, color="k", s=14, zorder=2)
ax.set_xticks(range(len(MODELS)), list(MODELS))
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("best normalized score (held-out)")
ax.set_title("Qwen2.5-1.5B: base vs three loss masks")
os.makedirs(f"{ROOT}/figures", exist_ok=True)
fig.tight_layout(); fig.savefig(f"{ROOT}/figures/palm_vs_arex.png", dpi=200); plt.show()

## 9. Output

The whole results folder is this version's output. This also makes a small summary zip to download from the Output tab.

In [ ]:
!cd {ROOT} && zip -qr /kaggle/working/mk33_train_palm_summary.zip comparison.md comparison.json figures data/palm_weights.jsonl \
    eval/*/report.md eval/*/summary.json teacher/report.md
!ls -la /kaggle/working